# Simple API connection to an MODEL SERVING ENDPOINT

In [1]:
from openai import OpenAI
import os

In [2]:
var_client = OpenAI(
                        api_key  = os.getenv("var_AI_API_sec", "null"),
                        base_url = os.getenv("var_end_point", "null")
                    )

In [3]:
while True:
    print("\n")
    var_user_prompt = input("Your prompt message here: ")
    print("################################################")
    if var_user_prompt == 'y_done':
        break
    var_response = var_client.chat.completions.create(
                                                        messages = [
                                                                        {
                                                                            "role": "system",
                                                                            #"content": "You are 3 year old child who doesnt know much"
                                                                            "content": "You are a general helpful question answer assistant , answer briefly"
                                                                        },
                                                                        {
                                                                            "role": "user",
                                                                            "content": var_user_prompt
                                                                        }
                                                                    ],
                                                        model="databricks-meta-llama-3-1-8b-instruct",
                                                        max_tokens=256
                                                    )

    print(var_response.choices[0].message.content)

Your prompt message here:  bye


################################################
Have a great day!




Your prompt message here:  y_done


################################################


# Unity catalog SETUP 

In [4]:
from databricks.sdk import WorkspaceClient
import os

In [5]:
w = WorkspaceClient(
                        host  = os.getenv("var_base_url", "null"),
                        token = os.getenv("var_AI_API_mn", "null")
                    )

In [6]:
display(w.dbutils.fs.ls('/Volumes/y_ws_250705/y_schema_for_ai/y_volume_for_rag/'))

[FileInfo(path='/Volumes/y_ws_250705/y_schema_for_ai/y_volume_for_rag/y_info_for_rag.txt', name='y_info_for_rag.txt', size=5358, modificationTime=1767528088000)]

In [ ]:
created_schema = w.schemas.create   ( 
                                        name         = "y_schema_for_AI",
                                        catalog_name = "y_ws_250705",
                                        comment      = "Created schema especially for AI related content"
                                    )

In [ ]:
from databricks.sdk.service.catalog import VolumeType
created_volume = w.volumes.create  (
                                        catalog_name = "y_ws_250705",
                                        schema_name  = "y_schema_for_AI",
                                        name         = "y_volume_for_RAG",
                                        volume_type  = VolumeType.MANAGED,
                                        comment      = "Volumes to store my RAG source PDFs"
                                    )

# Vector search endpoint

In [7]:
from databricks.vector_search.client import VectorSearchClient

In [8]:
os.environ['DATABRICKS_HOST'] = os.getenv("var_base_url", "null")
os.environ['DATABRICKS_TOKEN'] = os.getenv("var_AI_API_mn", "null")

In [9]:
VS_ENDPOINT_NAME = "y_vector_search_endpoint_20260329"
vsc = VectorSearchClient()

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [10]:
existing_endpoints = [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]
print(existing_endpoints)

[]


In [11]:
if VS_ENDPOINT_NAME not in existing_endpoints:
    print(f"Creating endpoint {VS_ENDPOINT_NAME}...")
    vsc.create_endpoint(name=VS_ENDPOINT_NAME,
                        endpoint_type="STANDARD",
                        budget_policy_id=os.getenv("var_serverless_policy_id_2","null")
                       )
else:
    print(f"Endpoint {VS_ENDPOINT_NAME} already exists.")

while vsc.get_endpoint(VS_ENDPOINT_NAME).get("endpoint_status", {}).get("state") != "ONLINE":
    print("Waiting for endpoint to come online...")
    time.sleep(30)

print("Endpoint is ONLINE.")

Creating endpoint y_vector_search_endpoint_20260329...
Endpoint is ONLINE.


In [15]:
# --- CONFIGURATION ---
CATALOG = "y_ws_250705"
SCHEMA = "y_schema_for_ai"
VOLUME_NAME = "y_volume_for_rag"
FILE_NAME = "y_info_for_rag.txt"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.y_tbl_for_chunking"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/{FILE_NAME}"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.y_test_index_1"

In [16]:

#### Create the index
vsc.create_delta_sync_index(
                                endpoint_name=VS_ENDPOINT_NAME,
                                source_table_name=TABLE_NAME,
                                index_name=INDEX_NAME,
                                pipeline_type='TRIGGERED', # Use 'CONTINUOUS' if you want real-time updates
                                primary_key="id",
                                embedding_source_column="content",
                                embedding_model_endpoint_name="databricks-bge-large-en" 
                            )

print(f"Index {INDEX_NAME} creation initiated. It will now begin embedding your Gwinzol data.")

Index y_ws_250705.y_schema_for_ai.y_test_index_1 creation initiated. It will now begin embedding your Gwinzol data.


# Similarity search :

In [17]:
index = vsc.get_index(endpoint_name=VS_ENDPOINT_NAME, index_name=INDEX_NAME)

In [32]:
query = "Tell me about the Gwinzol's "

In [33]:
results = index.similarity_search   (
                                        query_text=query,
                                        columns=["content"], # Return the text chunk
                                        num_results=3        # Get the top 2 most relevant chunks
                                    )

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [36]:
search_results = results.get('result', {}).get('data_array', [])
temp_dict={}
for i, res in enumerate(search_results):
    temp_dict[i] = res

print(temp_dict)
print('h')

[['vehicle. The Gwinzol also features a "Ghost-Mode" for track driving, where a holographic projection', 0.5465636], ['Mechanically, the Gwinzol is powered by a solid-state "Neutron-Flow" battery pack located in the', 0.54636735], ['driver telemetry. Each Gwinzol is assigned a unique "AI-Twin," a digital consciousness stored in', 0.5429043]]
h


# Actual RAG part 

In [31]:
context = "context is :"
for i, res in enumerate(search_results):
    context=context+res[0]

prompt = f"""You are a luxury automotive expert for Gwinzol. 
                    Use the following pieces of retrieved context to answer the question. 
                    If you don't know the answer based on the context, say you don't know. 
                    Keep the answer professional and exciting.

                    Context:
                    {context}

                    Question: 
                    {question}

                    Answer:"""

print (prompt)

You are a luxury automotive expert for Gwinzol. 
                    Use the following pieces of retrieved context to answer the question. 
                    If you don't know the answer based on the context, say you don't know. 
                    Keep the answer professional and exciting.

                    Context:
                    context is :vehicle. The Gwinzol also features a "Ghost-Mode" for track driving, where a holographic projectionMechanically, the Gwinzol is powered by a solid-state "Neutron-Flow" battery pack located in thedriver telemetry. Each Gwinzol is assigned a unique "AI-Twin," a digital consciousness stored in

                    Question: 
                    what is gwinzol

                    Answer:


In [38]:
while True:
    print("\n")
    query = input("Your prompt message here: ")
    print("################################################")
    results = index.similarity_search   (
                                            query_text=query,
                                            columns=["content"], # Return the text chunk
                                            num_results=3        # Get the top 2 most relevant chunks
                                        )
    search_results = results.get('result', {}).get('data_array', [])
    temp_dict={}
    for i, res in enumerate(search_results):
        temp_dict[i] = res
    
    prompt = f"""You are a luxury automotive expert for Gwinzol. 
                    Use the following pieces of retrieved context to answer the question. 
                    If you don't know the answer based on the context, say you don't know. 
                    Keep the answer professional and exciting.

                    Context is given in dictionary format from RAG:
                    {temp_dict}

                    Question: 
                    {query}

                    Answer:"""
    if query == 'y_done':
        break
    var_response = var_client.chat.completions.create(
                                                        messages = [
                                                                        {
                                                                            "role": "system",
                                                                            #"content": "You are 3 year old child who doesnt know much"
                                                                            "content": "You are a general helpful question answer assistant , answer briefly"
                                                                        },
                                                                        {
                                                                            "role": "user",
                                                                            "content": prompt
                                                                        }
                                                                    ],
                                                        model="databricks-meta-llama-3-1-8b-instruct",
                                                        max_tokens=256
                                                    )

    print(var_response.choices[0].message.content)

Your prompt message here:  hi


################################################
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Hello! It's a pleasure to connect with you. I'm excited to discuss the latest luxury automotive innovations from Gwinzol. How can I assist you in creating a truly personalized mobility experience today?




Your prompt message here:  what do you know about gwinzol


################################################
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
The Gwinzol - a marvel of modern automotive engineering! I can tell you that it's a remarkable vehicle powered by a cutting-edge "Neutron-Flow" battery pack, providing exceptional performance and efficiency. Additionally, its internal design showcases a mastery of advanced materials science and ergonomic innovation, ensuring a luxurious and comfortable driving experience.

Furthermore, the Gwinzol features a sophisticated four-wheel "Active-Pivot" setup, capable of executing a unique "Crab-Walk" maneuver, allowing for agile and precise handling on various terrains.

I'm thrilled to share my expertise on this extraordinary vehicle, and I'm confident that the Gwinzol will revolutionize the world of luxury driving!




Your prompt message here:  tell me about gwinzol in just one sentence


################################################
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
The Gwinzol is a high-performance luxury vehicle equipped with advanced features such as a rapid 80% replenishment capability and a state-of-the-art "Neutron-Flow" battery pack for superior power and efficiency.




Your prompt message here:  y_done


################################################
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
